# 01 - Data preparation and pronunciation-code discovery

All the offline work happens here. Afterwards training reads only memmaps, so
the GPU never waits on data.

| Stage | What it does | Time on a 4090 |
|---|---|---|
| A0 | manifest: scan, normalize, filter | 1 min |
| A1 | CTC forced alignment to word spans | about 35 min |
| A2 | self-supervised embeddings per word span | about 25 min |
| A3 | **discover pronunciation codes** | about 8 min |
| A4 | MARBERTv2 teacher cache and head | about 8 min |
| A5 | Mimi encode to RVQ codes | about 30 min |

Stage A3 is the novel part: it decides from audio alone which words have more
than one pronunciation. Nothing is hardcoded.

In [1]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"

# ---------------------------------------------------------------------------
# PICK YOUR EXPERIMENT HERE. This is the only line to change.
#
#   configs/exp0_small.yaml     2013 clips, ~2 GB   -> proves the pipeline,
#                                                     runs on a 6 GB GPU
#   configs/exp1_egyptian.yaml  15.6k clips, 68 h   -> the real run
# ---------------------------------------------------------------------------
CONFIG = "configs/exp1_egyptian.yaml"

from adaptts.utils.config import load_config
from adaptts.utils.logging_utils import setup_logging
setup_logging()
cfg = load_config(CONFIG)
print("repo   :", REPO)
print("config :", CONFIG, "->", cfg.name)
print("dataset:", cfg.paths.hf_dataset_id)

# The homographs named in the brief, read from the probe file so the notebooks
# never hardcode a word list of their own.
import json as _json
PROBE_WORDS = sorted({
    w for _s in _json.load(open("assets/probe_sentences.json", encoding="utf-8"))["sentences"]
    for w in _s["focus"].split(" / ") if w and w != "none"
})
print("probe  :", " ".join(PROBE_WORDS))

repo   : g:\Adaptive-TTS
config : configs/exp1_egyptian.yaml -> exp1_egyptian_homograph
dataset: ehabnegm/100-hour-Egyptian-dataset-single-speaker
probe  : الدول دول علم مصر


## Download the dataset

About 12 GB. To use a different corpus, change `paths.hf_dataset_id` and
`paths.dataset_dir` in the config.

In [2]:
os.environ.setdefault('HF_TOKEN', '')

In [ ]:
from huggingface_hub import snapshot_download

target = cfg.paths.dataset_dir
print("downloading to:", target)
snapshot_download(
    repo_id=cfg.paths.hf_dataset_id, repo_type="dataset",
    local_dir=target, max_workers=8,
)
print("done")

downloading to: g:\Adaptive-TTS\data\masri100h
12:56:39 info  httpx          HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
12:56:39 info  httpx          HTTP Request: GET https://huggingface.co/api/datasets/ehabnegm/100-hour-Egyptian-dataset-single-speaker/revision/main "HTTP/1.1 200 OK"


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 31566 files:   0%|          | 0/31566 [00:00<?, ?it/s]

12:57:12 info  httpx          HTTP Request: HEAD https://huggingface.co/datasets/ehabnegm/100-hour-Egyptian-dataset-single-speaker/resolve/03e4938f4c53b2ed6d2d36ca4dbf73175b1c43ee/clips/Ix_qkxx7A2U/Ix_qkxx7A2U_0043.txt "HTTP/1.1 307 Temporary Redirect"
12:57:12 info  httpx          HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ehabnegm/100-hour-Egyptian-dataset-single-speaker/03e4938f4c53b2ed6d2d36ca4dbf73175b1c43ee/clips%2FIx_qkxx7A2U%2FIx_qkxx7A2U_0043.txt?%2Fdatasets%2Fehabnegm%2F100-hour-Egyptian-dataset-single-speaker%2Fresolve%2F03e4938f4c53b2ed6d2d36ca4dbf73175b1c43ee%2Fclips%2FIx_qkxx7A2U%2FIx_qkxx7A2U_0043.txt=&etag=%22b533a44dfc3ac76e0b4783756a3d658fade49b21%22 "HTTP/1.1 200 OK"
12:57:12 info  httpx          HTTP Request: HEAD https://huggingface.co/datasets/ehabnegm/100-hour-Egyptian-dataset-single-speaker/resolve/03e4938f4c53b2ed6d2d36ca4dbf73175b1c43ee/clips/Ix_qkxx7A2U/Ix_qkxx7A2U_0044.txt "HTTP/1.1 307 Temporary Redirect"
12:57:12 info  httpx      

In [7]:
import glob, os

wavs = glob.glob(os.path.join(target, "clips", "**", "*.wav"), recursive=True)
parquet = glob.glob(os.path.join(target, "**", "*.parquet"), recursive=True)
meta = os.path.join(target, "metadata")
print("wav clips     :", len(wavs))
print("parquet shards:", len(parquet))
print("metadata dir  :", os.listdir(meta) if os.path.isdir(meta) else "none")
if parquet and not wavs:
    print()
    print("This is a parquet dataset. The manifest stage below unpacks the")
    print("embedded audio to wav once, so later stages never decode it again.")

wav clips     : 2013
parquet shards: 5
metadata dir  : none


## Stage A0 - manifest

Normalizes every transcript once, so alignment, the teacher and training all see
byte-identical strings.

In [8]:
!python scripts/preprocess.py --config $CONFIG --stage manifest

02:22:38 info  preprocess     AdapTTS preprocessing
02:22:38 info  preprocess     device: cuda
02:22:38 info  preprocess     configuration
02:22:38 info  preprocess     setting     value                                     
02:22:38 info  preprocess     ------------------------------------------------------
02:22:38 info  preprocess     experiment  exp0_small_egyptian                       
02:22:38 info  preprocess     dataset     g:\Adaptive-TTS\data\egy_small            
02:22:38 info  preprocess     cache       g:\Adaptive-TTS\cache\exp0                
02:22:38 info  preprocess     run dir     g:\Adaptive-TTS\runs\exp0                 
02:22:38 info  preprocess     codec       kyutai/mimi (8 quantizers)                
02:22:38 info  preprocess     teacher     aubmindlab/bert-base-arabertv02-twitter   
02:22:38 info  preprocess     aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:22:38 info  preprocess     precision   fp16                                      
02:22:38 in

In [9]:
import json, collections

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} utterances, {hours:.1f} hours")
print("splits:", collections.Counter(r["split"] for r in rows))
for r in rows[:3]:
    print(f'  [{r["duration"]:.1f}s] {r["text"][:70]}')

2012 utterances, 3.0 hours
splits: Counter({'train': 1853, 'dev': 80, 'test': 79})
  [5.1s] السلام عليكم ورحمه الله وبركاته اهلا وسهلا بكم فى حلقه جديده من بهدوء
  [5.0s] مع كريم حلقه النهارده هى فى مدخل رمضان و اصل الله سبحانه و تعالى
  [5.1s] لان كل واحد بقى بيخش يخبط بالزاويه اللي يعرفها فانا هخبط من زاويه شغلي


### Check the text normalization

Every transcript passed through the Egyptian normalizer. Numbers, dates and
Latin tokens should all be spoken words by now, with no digits left.

In [10]:
import json, random

rows = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
random.seed(0)
for r in random.sample(rows, min(8, len(rows))):
    print(f"[{r['duration']:5.1f}s] {r['text'][:100]}")

leftover = [r for r in rows if any(c.isdigit() for c in r["text"])]
print()
print(f"utterances still containing digits: {len(leftover)} of {len(rows)}")
for r in leftover[:3]:
    print("   ", r["text"][:100])

[  5.2s] اول فكره عايز اتكلم فيها هي فكره القلب والعقل انا ممكن تكون واحده من الناس
[  5.1s] في عندنا كتاب عملناه اسمه يستحي سوف نضع لكم اللينك
[  5.0s] خفت عادي جدا شويه نقح وراحوا لحاله انا في رمضان
[  5.4s] يعلمني مثلا الجدول بتاع العباده اللي في العمر اصدقائي
[  5.6s] هذه القصه من اولها الراجل اللي فجاه
[  5.1s] ده سنه بس مش لازم في السيده زينب في المكان الفلاني ولا على العربيه
[  5.0s] فانت لو فكرت مثلا انك كل خمس دقايق
[  5.0s] كل اللى فكروا كده اكتئبوا ليه دخلوا خبطوا في الحيطه اتكل على نفسه قاعد يرهق

utterances still containing digits: 0 of 2012


## Stage A1 - CTC forced alignment

Finds the time span of every word with no pronunciation lexicon. The Viterbi
alignment is implemented in-repo, so there is no Montreal Forced Aligner
dependency.

In [11]:
!python scripts/preprocess.py --config $CONFIG --stage align

02:22:54 info  preprocess     AdapTTS preprocessing
02:22:54 info  preprocess     device: cuda
02:22:54 info  preprocess     configuration
02:22:54 info  preprocess     setting     value                                     
02:22:54 info  preprocess     ------------------------------------------------------
02:22:54 info  preprocess     experiment  exp0_small_egyptian                       
02:22:54 info  preprocess     dataset     g:\Adaptive-TTS\data\egy_small            
02:22:54 info  preprocess     cache       g:\Adaptive-TTS\cache\exp0                
02:22:54 info  preprocess     run dir     g:\Adaptive-TTS\runs\exp0                 
02:22:54 info  preprocess     codec       kyutai/mimi (8 quantizers)                
02:22:54 info  preprocess     teacher     aubmindlab/bert-base-arabertv02-twitter   
02:22:54 info  preprocess     aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:22:54 info  preprocess     precision   fp16                                      
02:22:54 in

## Stages A2 and A3 - span embeddings and code discovery

The heart of the system. Each occurrence of each word type is embedded with a
self-supervised speech model, then we ask whether those embeddings form one
cluster or several. A split is accepted only when it is reproducible under
bootstrap resampling, geometrically clean, and acoustically well separated.

In [12]:
!python scripts/preprocess.py --config $CONFIG --stage discover

02:22:59 info  preprocess     AdapTTS preprocessing
02:22:59 info  preprocess     device: cuda
02:22:59 info  preprocess     configuration
02:22:59 info  preprocess     setting     value                                     
02:22:59 info  preprocess     ------------------------------------------------------
02:22:59 info  preprocess     experiment  exp0_small_egyptian                       
02:22:59 info  preprocess     dataset     g:\Adaptive-TTS\data\egy_small            
02:22:59 info  preprocess     cache       g:\Adaptive-TTS\cache\exp0                
02:22:59 info  preprocess     run dir     g:\Adaptive-TTS\runs\exp0                 
02:22:59 info  preprocess     codec       kyutai/mimi (8 quantizers)                
02:22:59 info  preprocess     teacher     aubmindlab/bert-base-arabertv02-twitter   
02:22:59 info  preprocess     aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:22:59 info  preprocess     precision   fp16                                      
02:22:59 in


A3 labelling occurrences  : 100%|██████████| 9/9 [00:00<00:00, 11.40word/s]


### What did it discover?

This is the moment of truth. Words like the ones named in the brief should
appear with more than one reading.

In [13]:
from adaptts.data.discovery import PronunciationLexicon

lex = PronunciationLexicon.load(cfg.paths.lexicon_path)
amb = lex.ambiguous_words
print(f"{len(amb)} ambiguous word types discovered")
print()

entries = sorted((lex.entries[w] for w in amb), key=lambda e: -e.occurrence_count)
header = f"{'word':<18}{'codes':>6}{'occ':>7}{'stab':>8}{'sil':>8}{'sep':>8}  counts"
print(header)
print("-" * 74)
for e in entries[:40]:
    print(f"{e.word:<18}{e.n_codes:>6}{e.occurrence_count:>7}{e.stability:>8.2f}"
          f"{e.silhouette:>8.2f}{e.separation:>8.2f}  {e.counts}")

9 ambiguous word types discovered

word               codes    occ    stab     sil     sep  counts
--------------------------------------------------------------------------
نفسه                   2     25    0.74    0.20    0.57  [21, 4]
وده                    2     24    0.81    0.19    0.60  [15, 9]
وفي                    2     23    0.87    0.20    0.59  [16, 7]
ومن                    2     19    0.81    0.18    0.65  [16, 3]
الترابيزه              2     17    0.76    0.12    0.64  [14, 3]
تانيه                  2     17    0.77    0.18    0.57  [11, 6]
ومش                    2     17    0.82    0.17    0.56  [9, 8]
حلو                    2     15    0.73    0.16    0.68  [12, 3]
فاتت                   2     14    0.72    0.11    0.56  [10, 4]


In [14]:
# Check the specific homographs named in the project brief.
for w in PROBE_WORDS:
    k = lex.n_codes(w)
    print(f"{w:<10} -> {k} code(s)   " + ("AMBIGUOUS" if k > 1 else "single reading"))

الدول      -> 1 code(s)   single reading
دول        -> 1 code(s)   single reading
علم        -> 1 code(s)   single reading
مصر        -> 1 code(s)   single reading


In [15]:
# Read the sentences behind each code. This is how you confirm the clusters
# track meaning rather than recording conditions.
import json, collections, os

labels = json.load(open(os.path.join(cfg.paths.cache_dir, "code_labels.json"), encoding="utf-8"))
manifest = [json.loads(l) for l in open(cfg.paths.manifest_path, encoding="utf-8")]
by_uid = {r["uid"]: r["text"] for r in manifest}

WORD = PROBE_WORDS[0]      # change to inspect any discovered homograph
groups = collections.defaultdict(list)
for key, code in labels.items():
    uid, widx = key.split(chr(9))
    text = by_uid.get(uid, "")
    words = text.split()
    if int(widx) < len(words) and words[int(widx)] == WORD:
        groups[code].append(text)

for code in sorted(groups):
    print()
    print(f"=== {WORD}  code {code}  ({len(groups[code])} occurrences) ===")
    for t in groups[code][:6]:
        print("   ", t[:95])

If the sentences under each code share a meaning, discovery worked. If they look
mixed, raise `discovery.min_separation` or `discovery.stability_threshold` in the
config and rerun this stage with `--force`.

## Stage A4 - teacher cache

One frozen MARBERTv2 pass over the corpus, cached as fp16. The teacher never
runs again, which is the main reason training is cheap.

In [16]:
!python scripts/preprocess.py --config $CONFIG --stage teacher

02:23:28 info  preprocess     AdapTTS preprocessing
02:23:28 info  preprocess     device: cuda
02:23:28 info  preprocess     configuration
02:23:28 info  preprocess     setting     value                                     
02:23:28 info  preprocess     ------------------------------------------------------
02:23:28 info  preprocess     experiment  exp0_small_egyptian                       
02:23:28 info  preprocess     dataset     g:\Adaptive-TTS\data\egy_small            
02:23:28 info  preprocess     cache       g:\Adaptive-TTS\cache\exp0                
02:23:28 info  preprocess     run dir     g:\Adaptive-TTS\runs\exp0                 
02:23:28 info  preprocess     codec       kyutai/mimi (8 quantizers)                
02:23:28 info  preprocess     teacher     aubmindlab/bert-base-arabertv02-twitter   
02:23:28 info  preprocess     aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:23:28 info  preprocess     precision   fp16                                      
02:23:28 in

## Stage A5 - Mimi codec encoding

Every clip becomes 8 RVQ streams at 12.5 Hz. A 10 second clip is 125 frames,
which is why generation is fast on a CPU.

In [17]:
!python scripts/preprocess.py --config $CONFIG --stage codec

02:23:33 info  preprocess     AdapTTS preprocessing
02:23:33 info  preprocess     device: cuda
02:23:33 info  preprocess     configuration
02:23:33 info  preprocess     setting     value                                     
02:23:33 info  preprocess     ------------------------------------------------------
02:23:33 info  preprocess     experiment  exp0_small_egyptian                       
02:23:33 info  preprocess     dataset     g:\Adaptive-TTS\data\egy_small            
02:23:33 info  preprocess     cache       g:\Adaptive-TTS\cache\exp0                
02:23:33 info  preprocess     run dir     g:\Adaptive-TTS\runs\exp0                 
02:23:33 info  preprocess     codec       kyutai/mimi (8 quantizers)                
02:23:33 info  preprocess     teacher     aubmindlab/bert-base-arabertv02-twitter   
02:23:33 info  preprocess     aligner     MahmoudAshraf/mms-300m-1130-forced-aligner
02:23:33 info  preprocess     precision   fp16                                      
02:23:33 in

## Verify the cache is complete

In [18]:
from adaptts.data.dataset import AdapTTSDataset, collate
from adaptts.text.vocab import CharVocab

vocab = CharVocab.load(cfg.paths.charvocab_path)
ds = AdapTTSDataset(cfg, "train", vocab, lex, need_codes=True, need_teacher=True)
b = collate(
    [ds[i] for i in range(4)], vocab.pad_id, cfg.discovery.max_codes_per_word,
    cfg.audio.n_quantizers, cfg.teacher.hidden_size,
)
for k, v in b.items():
    print(f"  {k:<20} {tuple(v.shape)}  {v.dtype}")
print()
print("ambiguous words in this batch:", int((b["n_codes"] > 1).sum()))
ds.close()
print()
print("Data is ready. Continue to 02_train_context.ipynb")

02:23:37 info  dataset        dataset[train]: 1853 utterances, length range 63..145
  char_ids             (4, 72)  torch.int64
  word_index           (4, 72)  torch.int64
  char_padding_mask    (4, 72)  torch.bool
  n_codes              (4, 14)  torch.int64
  code_target          (4, 14)  torch.int64
  word_mask            (4, 14)  torch.bool
  pc_per_char          (4, 72)  torch.int64
  n_frames             (4,)  torch.int64
  codes                (4, 65, 8)  torch.int64
  frame_mask           (4, 65)  torch.bool
  teacher_hidden       (4, 14, 768)  torch.float32

ambiguous words in this batch: 0

Data is ready. Continue to 02_train_context.ipynb
